# nbkit — live demo

Every output below was produced by running this notebook against a real IPython
kernel. Nothing is transcribed by hand.

What is real here and what is stood in for:

| | status |
|---|---|
| L0 read | real — reads this notebook off disk |
| L1 write | real payload emitted; no frontend attached to apply it |
| L2 model | real HTTPS calls, real fallback — **DeepSeek standing in for the campus endpoints** |
| L3 triggers | real — hooks fire on real cell execution |

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath("."))

# The endpoints are not in hand yet, so this demo stands DeepSeek in for
# DiveAI. It exercises the same code path — an OpenAI-compatible POST.
for line in open("../.env"):
    if "=" in line and not line.lstrip().startswith("#"):
        k, v = line.strip().split("=", 1)
        os.environ.setdefault(k, v)

# The stand-in endpoints below are local servers; keep them off any proxy.
os.environ["no_proxy"] = "127.0.0.1,localhost"

import nbkit, nbkit_ask, nbkit_hooks
print("nbkit loaded — kernel:", type(nbkit.shell()).__name__)

nbkit loaded — kernel: ZMQInteractiveShell


## L0 — reading the notebook

In [2]:
cells = nbkit.get_cells()
print(f"{len(cells)} cells found in this notebook")
for i, c in enumerate(cells[:6]):
    first = nbkit.cell_source(c).split("\n")[0][:58]
    print(f"  [{i}] {c['cell_type']:<9} {first}")

30 cells found in this notebook
  [0] markdown  # nbkit — live demo
  [1] code      import os, sys
  [2] markdown  ## L0 — reading the notebook
  [3] code      cells = nbkit.get_cells()
  [4] code      # The source of the cell running right now — from IPython'
  [5] code      # Which source answered? Colab has no file, this machine h


In [ ]:
# The source of the cell running right now — from IPython's input history,
# no notebook file involved.
print(repr(nbkit.get_current_cell()[:70]))

In [ ]:
# Which source answered? Colab has no file, this machine has no
# jupyter-mcp-cli, so it fell to the disk glob.
print("colab :", len(nbkit._cells_from_colab()))
print("mcp   :", len(nbkit._cells_from_mcp()))
print("disk  :", len(nbkit._cells_from_disk()))

In [ ]:
# Position in the notebook, needed by anything that reads "the cell above me".
# Returns None rather than guessing when the disk copy has drifted.
print("current_index() ->", nbkit.current_index())
print("cells above     ->", [s[:40] for s in nbkit.get_cells_before(1)])

In [ ]:
# Errors, formatted the way a prompt wants them: type and message, no frames.
try:
    {"a": 1}["b"]
except KeyError as e:
    print(repr(nbkit.format_error(e)))

## L1 — writing to the notebook

The kernel cannot reach into the document. It sends the frontend a
`set_next_input` payload and asks. `nbconvert` has no frontend, so below we
inspect the payload instead of watching a cell change — which is the honest
version of this demo, and shows exactly what a live frontend would receive.

In [ ]:
ip = nbkit.shell()
ip.payload_manager.clear_payload()

nbkit.insert_cell_below("# written by nbkit\nprint('hello from a generated cell')")

for p in ip.payload_manager.read_payload():
    print("source :", p["source"])
    print("replace:", p["replace"], " <- False means 'new cell below', True means 'overwrite this one'")
    print("text   :", repr(p["text"]))
ip.payload_manager.clear_payload()

## L2 — the model layer, and the fallback that matters

One class is ~500 students against a ~30-concurrent endpoint. `ask()` tries the
better model first and falls through when it cannot answer. Below, the first
endpoint is genuinely broken and the second genuinely answers.

In [ ]:
# Endpoint 1: nothing is listening on port 9. A real connection failure.
# Endpoint 2: a real OpenAI-compatible server that really answers.
os.environ["NBKIT_LITELLM_BASE_URL"] = "http://127.0.0.1:9/v1"
os.environ["NBKIT_LITELLM_MODEL"]    = "qwen3.6-27b"
os.environ["NBKIT_DIVE_BASE_URL"]    = "https://api.deepseek.com/v1"
os.environ["NBKIT_DIVE_MODEL"]       = "deepseek-chat"
os.environ["NBKIT_DIVE_API_KEY"]     = os.environ["DEEPSEEK_API_KEY"]

for e in nbkit_ask.endpoints():
    print(f"{e.name:<8} {e.model:<14} {e.base_url}")

In [ ]:
import time
t0 = time.time()
answer = nbkit_ask.ask("In one short sentence: what does Python's zip() do?")
print(f"answered in {time.time()-t0:.1f}s after the first endpoint refused\n")
print(answer)

In [ ]:
# A saturated endpoint, for real: a local server that returns 429 to everything.
# ask() retries it once, then gives up on it and falls through.
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer

hits = []

class Busy(BaseHTTPRequestHandler):
    def do_POST(self):
        hits.append(time.time())
        self.send_response(429); self.end_headers(); self.wfile.write(b"busy")
    def log_message(self, *a): pass

srv = HTTPServer(("127.0.0.1", 8731), Busy)
threading.Thread(target=srv.serve_forever, daemon=True).start()

os.environ["NBKIT_LITELLM_BASE_URL"] = "http://127.0.0.1:8731/v1"
t0 = time.time()
answer = nbkit_ask.ask("Reply with exactly: fell through")
srv.shutdown()

print(f"the busy endpoint was hit {len(hits)} times (1 try + 1 retry)")
print(f"total {time.time()-t0:.1f}s, including the {nbkit_ask.BUSY_RETRY_WAIT}s backoff")
print("answer:", answer)

In [ ]:
# Streaming, so a slow model does not look like a frozen cell.
os.environ["NBKIT_LITELLM_BASE_URL"] = "http://127.0.0.1:9/v1"
pieces = []
for piece in nbkit_ask.stream("Count from 1 to 5, comma separated. Nothing else."):
    pieces.append(piece)
print(f"{len(pieces)} chunks arrived separately:")
print("".join(pieces))

In [ ]:
# Every endpoint down: raises, and names each one. Not a silent empty string.
os.environ["NBKIT_LITELLM_BASE_URL"] = "http://127.0.0.1:9/v1"
os.environ["NBKIT_DIVE_BASE_URL"]    = "http://127.0.0.1:9/v1"
try:
    nbkit_ask.ask("anything")
except RuntimeError as e:
    print(e)
os.environ["NBKIT_DIVE_BASE_URL"] = "https://api.deepseek.com/v1"

## L3 — triggers

The conceptual jump: from here the package runs without being invoked.

In [ ]:
seen = []

@nbkit_hooks.on_cell_run
def watch(source):
    seen.append(source.split("\n")[0][:40])

print("registered. run a couple of cells...")

In [ ]:
x = 6 * 7

In [ ]:
print("hook saw:", seen)

In [ ]:
# Re-running the registration cell is the single most common thing a student
# does. Register three times, still one hook.
for _ in range(3):
    @nbkit_hooks.on_cell_run
    def watch(source):
        seen.append("dup")

print("hooks attached:", len(nbkit.shell().events.callbacks["post_run_cell"]))

In [ ]:
# A hook that raises must not break the student's unrelated code.
@nbkit_hooks.on_cell_run
def broken(source):
    raise RuntimeError("this hook is broken on purpose")

In [ ]:
2 + 2

### The error whisperer — notebook 3, end to end

Reactive, and it is the whole package: one hook, one prompt, one rendered
answer. This is the point at which a student has shipped something real.

In [ ]:
nbkit_hooks.clear_hooks()

@nbkit_hooks.on_cell_error
def whisper(source, error):
    reply = nbkit_ask.ask(
        f"A beginner hit this error:\n{error}\n\nTheir code:\n{source}\n\n"
        "Ask ONE short guiding question. Never give the answer.",
        system="You are a patient coding mentor. One question, no answers.",
        max_tokens=80,
    )
    nbkit.show_md(f"> 🏛️ **{reply.strip()}**")

print("error whisperer armed")

In [ ]:
prices = {"apple": 3, "pear": 5}
total = 0
for fruit in ["apple", "pear", "plum"]:
    total += prices[fruit]

In [ ]:
nbkit_hooks.clear_hooks()
print("hooks cleared")

## L1, for real — run this one in a browser

The cell above inspects the payload and then throws it away, which is why it is
safe under `nbconvert`. This one does not throw it away. Run it in JupyterLab
and a new cell appears below, already filled in; run it headless and nothing
happens, because there is no frontend listening.

This is the only part of the demo that batch execution genuinely cannot show.

In [ ]:
nbkit.insert_cell_below(
    "# ← this cell was written by nbkit, not typed\n"
    "print('a package just edited your notebook')"
)

In [ ]:
# And the destructive one: overwrite the cell you are running.
# Run it twice and note that nothing further changes — the write is idempotent
# only because the text is constant. Real packages have to arrange that
# themselves, which is notebook 2's whole lesson.
nbkit.replace_current_cell(
    "# nbkit overwrote this cell.\n"
    "# The original source is gone — Ctrl+Z in the notebook brings it back.\n"
    "print('overwritten')"
)